In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO
from urllib.parse import urljoin
from urllib.parse import urlparse
import os
from pathlib import Path
import pdfplumber

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_pdf_dir = project_root / "data" / "raw" / "pdfs"
reconstructed_pdf_dir = project_root / "data" / "intermediate" / "reconstructed_pdfs"
extracted_dir = project_root / "data" / "extracted"
processed_dir = project_root / "data" / "processed"
final_dir = project_root / "data" / "final"

raw_pdf_dir.mkdir(parents=True, exist_ok=True)
extracted_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
final_dir.mkdir(parents=True, exist_ok=True)


In [3]:
## %pip install --upgrade certifi requests urllib3

Note: you may need to restart the kernel to use updated packages.


In [2]:
url = "https://www.mumbwacouncil.gov.zm/"

In [3]:
response = requests.get(url,timeout=30 ,verify=False)

C:\Users\Moses Chaswala\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [4]:
print(response.status_code)
print(response.url)
print(response.text[:2000])

200
https://www.mumbwacouncil.gov.zm/
<!DOCTYPE html>
<html lang="en-US">
<head>
	<meta charset="UTF-8">
	<meta name="viewport" content="width=device-width, initial-scale=1.0, viewport-fit=cover" />		<title>Mumbwa Town Council &#8211; Mumbwa</title>
<meta name='robots' content='max-image-preview:large' />
<link rel='dns-prefetch' href='//s.w.org' />
<link rel="alternate" type="application/rss+xml" title="Mumbwa Town Council &raquo; Feed" href="https://www.mumbwacouncil.gov.zm/?feed=rss2" />
<link rel="alternate" type="application/rss+xml" title="Mumbwa Town Council &raquo; Comments Feed" href="https://www.mumbwacouncil.gov.zm/?feed=comments-rss2" />
<script>
window._wpemojiSettings = {"baseUrl":"https:\/\/s.w.org\/images\/core\/emoji\/13.1.0\/72x72\/","ext":".png","svgUrl":"https:\/\/s.w.org\/images\/core\/emoji\/13.1.0\/svg\/","svgExt":".svg","source":{"concatemoji":"https:\/\/www.mumbwacouncil.gov.zm\/wp-includes\/js\/wp-emoji-release.min.js?ver=5.9"}};
/*! This file is auto-generate

In [5]:
print("<table" in response.text.lower())

False


In [6]:
soup = BeautifulSoup(response.text, "html.parser")

print("Tables:", len(soup.find_all("table")))
print("Scripts:", len(soup.find_all("script")))
print("Links:", len(soup.find_all("a")))

Tables: 0
Scripts: 50
Links: 80


In [ ]:
# %pip install --upgrade certifi requests urllib3


In [10]:

soup = BeautifulSoup(response.text, "html.parser")

print("Tables found:", len(soup.find_all("table")))
print("Links found:", len(soup.find_all("a")))
print("Scripts found:", len(soup.find_all("script")))

Tables found: 0
Links found: 80
Scripts found: 50


In [7]:
links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = urljoin(url, link["href"])

    links.append({
        "text": text,
        "url": href
    })

links_df = pd.DataFrame(links)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(links_df)

,text,url
0,,https://www.mumbwacouncil.gov.zm/
1,Home,https://www.mumbwacouncil.gov.zm/
2,About,https://www.mumbwacouncil.gov.zm/#
3,Who we are,https://www.mumbwacouncil.gov.zm/?page_id=118
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
6,Civic Leaders,https://www.mumbwacouncil.gov.zm/?page_id=2868
7,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
8,Nangoma,https://www.mumbwacouncil.gov.zm/?page_id=2873
9,Mumbwa Central,https://www.mumbwacouncil.gov.zm/?page_id=2871


In [5]:
tracker_url = "https://www.mumbwacouncil.gov.zm/?page_id=932"

tracker_response = requests.get(
    tracker_url,
    verify=False,
    timeout=30
)

tracker_soup = BeautifulSoup(tracker_response.text, "html.parser")

print("Status:", tracker_response.status_code)
print("Tables:", len(tracker_soup.find_all("table")))
print("Links:", len(tracker_soup.find_all("a")))

C:\Users\Moses Chaswala\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
Tables: 0
Links: 114


In [6]:
tracker_links = []

for link in tracker_soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = urljoin(tracker_url, link["href"])

    tracker_links.append({
        "text": text,
        "url": href
    })

tracker_links_df = pd.DataFrame(tracker_links)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(tracker_links_df)

,text,url
0,,https://www.mumbwacouncil.gov.zm/
1,Home,https://www.mumbwacouncil.gov.zm/
2,About,https://www.mumbwacouncil.gov.zm/?page_id=932#
3,Who we are,https://www.mumbwacouncil.gov.zm/?page_id=118
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
6,Civic Leaders,https://www.mumbwacouncil.gov.zm/?page_id=2868
7,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
8,Nangoma,https://www.mumbwacouncil.gov.zm/?page_id=2873
9,Mumbwa Central,https://www.mumbwacouncil.gov.zm/?page_id=2871


In [7]:
cdf_pdfs = tracker_links_df[
    tracker_links_df["url"].str.contains(
        r"\.pdf",
        case=False,
        na=False
    )
].copy()

display(cdf_pdfs)

,text,url
66,2025 Approved Community Projects-Mumbwa Central,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects-Mumbwa-Central.pdf
67,2025 Proposed Community Projects-Mumbwa Central,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf
69,2025 Approved CDF Skills Development Bursaries for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf
71,2025 Approved CDF Secondary Boarding School Bursaries for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf
73,2025 Approved CDF Empowerment Grants for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf
94,2025 Approved Community Projects-Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects-Nangoma-Constituency.pdf
95,2025 Proposed Community Projects-Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA.pdf
97,2025 Approved CDF Empowerment Grants for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf
100,2025 Approved CDF Skills Development Bursaries for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf
102,2025 Approved CDF Secondary Boarding School Bursaries for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf


In [ ]:
mumbwa_cdf_pdfs = raw_pdf_dir
download_folder = raw_pdf_dir
os.makedirs(download_folder, exist_ok=True)

for _, row in cdf_pdfs.iterrows():

    url = row["url"]

    filename = os.path.basename(
        urlparse(url).path
    )

    filepath = os.path.join(
        download_folder,
        filename
    )

    print(f"Downloading: {filename}")

    pdf_response = requests.get(
        url,
        verify=False,
        timeout=100
    )

    print("Status:", pdf_response.status_code)

    if pdf_response.status_code == 200:
        with open(filepath, "wb") as f:
            f.write(pdf_response.content)

        print("Saved:", filepath)
    else:
        print("FAILED:", url)

    print("-" * 80)


In [ ]:
pdf_files = [
    str(pdf_file)
    for pdf_file in sorted(mumbwa_cdf_pdfs.glob("*.pdf"))
]

print("PDFs found:", len(pdf_files))

for pdf in pdf_files:
    print(os.path.basename(pdf))


In [ ]:
for pdf_file in pdf_files:

    print("\n" + "#" * 100)
    print(os.path.basename(pdf_file))
    print("#" * 100)

    with pdfplumber.open(pdf_file) as pdf:

        total_tables = 0

        for page_number, page in enumerate(pdf.pages, start=1):

            tables = page.extract_tables()

            if tables:
                print(
                    f"Page {page_number}: "
                    f"{len(tables)} table(s)"
                )

                total_tables += len(tables)

        print("TOTAL TABLES:", total_tables)

In [ ]:
first_pdf = pdf_files[0]

with pdfplumber.open(first_pdf) as pdf:

    print("File:", os.path.basename(first_pdf))
    print("Pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages[:3], start=1):

        text = page.extract_text()

        print("\n" + "=" * 80)
        print("PAGE", page_number)
        print("=" * 80)

        if text:
            print(text[:5000])
        else:
            print("NO TEXT FOUND")


In [ ]:
pdf1_path = raw_pdf_dir / "2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf"


In [18]:
all_tables = []

with pdfplumber.open(pdf1_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        table = page.extract_tables()[0]

        # Page 1 has title + blank row before headers
        header_row = 2 if page_num == 1 else 0
        data_start = 3 if page_num == 1 else 1

        df_page = pd.DataFrame(
            table[data_start:],
            columns=table[header_row]
        )
          
        all_tables.append(df_page)

df = pd.concat(all_tables, ignore_index=True)
df


,No.,Name of Group,Type,District,Constituency,Ward,Zone,Contact Person,Type of Venture,Sector
0,1,Intalisho Women's Club,Women,Mumbwa,Nangoma,Matala,Kantengwa,Markson Ngweluka,Gardening,Agriculture
1,2,Luyandano Multi-purpose Co-operative,Community,Mumbwa,Nangoma,Matala,Kantengwa,Prosper Malupande,Gardening,Agriculture
2,3,Kunumfwana Women's Club,Women,Mumbwa,Nangoma,Matala,Kantengwa,Rhoda Lumashi,Gardening,Agriculture
3,4,Kayuni Multi-Purpose Co-operative,Community,Mumbwa,Nangoma,Matala,Kantengwa,Chola Willard,Goat Keeping,Livestock
4,5,Faith Chibwe Club,Community,Mumbwa,Nangoma,Matala,Kantengwa,Elizabeth Musonko,Goat Keeping,Livestock
5,6,Twazuma Youth Club,Youth,Mumbwa,Nangoma,Matala,Kantengwa,Clever Kakwele,Goat Keeping,Livestock
6,7,Luili S.D.A Church Multi-purpose\nCooperative,Community,Mumbwa,Nangoma,Matala,Luili,Filner Kantini,Farming,Agriculture
7,8,Smart Climate Muliti - Purpose Cooperative,Community,Mumbwa,Nangoma,Matala,Luili,Weagan Muleka,Cattle Keeping,Agriculture
8,9,Makuyu Aboombe Youth Club,Youth,Mumbwa,Nangoma,Chisalu,Chilonga,Havi Mweemba,Goats,Livestock
9,10,Prefa Youth Club,Youth,Mumbwa,Nangoma,Chisalu,Chilonga,Fabian Mweendela,Goats,Livestock


In [ ]:
pdf2_path = raw_pdf_dir / "2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf"


In [ ]:
all_tables = []

with pdfplumber.open(pdf2_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        table = page.extract_tables()[0]

        # Page 1 has title + blank row before headers
        header_row = 2 if page_num == 1 else 0
        data_start = 3 if page_num == 1 else 1

        df_page = pd.DataFrame(
            table[data_start:],
            columns=table[header_row]
        )
          
        all_tables.append(df_page)

df2 = pd.concat(all_tables, ignore_index=True)
df2
